# 3. MESA Finetune

## Prerequisites

- MESA Runtime credentials

## Overview

Finetune sits alongside datagen as the second 'build' component of MESA (below). Much of finetune is handy middleware for interacting with underlying libraries (e.g. a HuggingFace estimator via AWS SageMaker, or MLX on Apple Silicon [bare metal]), including handling training data download, storing canonical training configurations, error recovery, and model tag/upload/publish.

<p align="center">
  <img src="https://londonaicentre.github.io/MESA-Build/repo.png" width="700" alt="Flowchart of the document generation pipeline">
</p>

> 📓 As per our running example, we will download and use the training pairs generated in the datagen notebook, and use these to train a small schema standardisation model for the oncology domain. Before that, an overview is provided of the wider training (and inference) architecture also provided by MESA, which can be skipped.

## (Optional) Training (and inference) architecture

Runtime is an in-house orchestration tool to support GPU-based tasks such as inference and training. Runtime leverages GitHub's existing workflow orchestration tool (shown below), with jobs run on in-house compute resources (e.g. a Mac Studio) via remote runners. Each job involves the use of a _software_ runner, which packages up relevant logic: in this case the fine-tuning logic provided by the finetune package.

<p align="center">
  <img src="https://londonaicentre.github.io/mesa-runtime/architecture.svg" width="700" alt="Diagram showing MESA training and inference architecture">
</p>

<p align="center">
  <img src="https://londonaicentre.github.io/mesa-runtime/frontend/ui.png" width="700" alt="Screenshot showing MESA runtime UI">
</p>

Finetuning jobs can be initiated by an admin on GitHub directly, via a custom Runtime UI (above) or via a python library. Build pipelines for existing models (e.g. `OncoQwen`) leverage this library, and it will be used in the example below.

## Steps

1. Import some utility classes to help us work with the library.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

2. Import the MESA runtime library, and create an object of its primary class with our credentials. 

In [ ]:
from uuid import UUID
from mesa_runtime import MesaRuntime
runtime: MesaRuntime = MesaRuntime(
    os.environ["LLM__MESA_RUNTIME__BASE_URL"],
    os.environ["LLM__MESA_RUNTIME__USERNAME"],
    os.environ["LLM__MESA_RUNTIME__PASSWORD"],
)

3. List the training data we have available to work with. We should see the batch we uploaded in the datagen step.

In [ ]:
runtime.list_training_data()

4. We can now start our finetuning job on the remote infrastructure (default: Mac Studio). We supply: 

- The name of the training data in our bucket (Step 3)
- The name of the model we're training
- The schema we've generated our training pairs for (with the word 'schema' truncated)
- A description
- A version for our model
- The configuration for our fine-tuning job -- [config.yaml](config.yaml), which can be inspected if you wish to -- whose values are merged over the stock configuration
- A flag to indicate that we want to push our model somewhere publicly accessible for the next stage of our example

> ❗Finetuning is a slow-running operation. If you wish to, this step can be skipped and you can move to the next notebook.

In [ ]:
run_id: UUID = runtime.train(
    "20260706-205744_oncotest-batch",
    "oncotest",
    "onco",
    "Test oncology model",
    "0.1.0",
    "config.yaml",
    True,
)

5. We can check the status of the job by passing the run id output from the previous command. Once it reads as complete, we're ready to move to the [next notebook](../deploy_and_runner/deploy_and_runner.ipynb).

In [ ]:
runtime.get_run(run_id)

⭐ Distilling the capabilities of large models into small, portable ones ('transfer learning') by fine-tuning on standardised versions of highly realistic synthetic documents is another key contribution of MESA.

## Alternative: AWS SageMaker

Steps 2 to 5 can instead be run on AWS SageMaker, using the finetune library directly, which needs a SageMaker execution role (`SAGEMAKER_EXECUTION_ROLE`) as well as access to our bucket. Install the library with `uv sync --group sagemaker`, then run the cells below in place of steps 2 to 5. We supply the schema and prompt builder that describe our training pairs, the training data from step 3, the same configuration, our bucket, region and role, and the name and description of our model.

>❗SageMaker training is billable for as long as the job runs.

In [ ]:
from finetune import HuggingFaceLoRATrainer
from oncoschema import Schema
from oncoschema.prompt_builder import PromptBuilder
trainer: HuggingFaceLoRATrainer = HuggingFaceLoRATrainer(
    Schema,
    PromptBuilder(),
    ["20260706-205744_oncotest-batch"],
    "config.yaml",
    {
        "bucket": os.environ["BUCKET"],
        "region": "eu-west-2",
        "role": os.environ["SAGEMAKER_EXECUTION_ROLE"],
    },
    "oncotest",
    "Test oncology model",
)

In [ ]:
job_name: str = trainer.launch_job(trainer.prepare_data())

Monitor the job in the AWS SageMaker console. Once its status is `Completed`, merge the trained adapter into the base model and publish it with our model's version, pushing it somewhere publicly accessible as before.

In [ ]:
trainer.post_process(trainer.build_model_card(0, 1, 0), trainer.s3_output_path, job_name, True)